In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import euclidean_distances

In [2]:
# ---------------------------------------------------------
# Step 1: Create Historical Customer Support Cases
# ---------------------------------------------------------

data = {
    "Issue_Type": [
        "Login Problem",
        "Payment Failure",
        "Login Problem",
        "Product Damage",
        "Payment Failure",
        "Delivery Delay",
        "Product Damage",
        "Login Problem"
    ],
    "Product": [
        "Mobile App",
        "Subscription",
        "Web Portal",
        "Laptop",
        "Mobile App",
        "Laptop",
        "Mobile App",
        "Web Portal"
    ],
    "Priority": [
        "High",
        "High",
        "Medium",
        "High",
        "Medium",
        "Low",
        "Medium",
        "High"
    ],
    "Customer_Type": [
        "Regular",
        "Premium",
        "Regular",
        "Premium",
        "Regular",
        "Regular",
        "Premium",
        "Premium"
    ],
    "Solution": [
        "Reset password and verify account",
        "Retry payment and verify payment details",
        "Reset password and clear browser cache",
        "Replace damaged product",
        "Verify card details and retry payment",
        "Track shipment and inform customer",
        "Initiate product replacement",
        "Reset password and verify account"
    ]
}

df = pd.DataFrame(data)

print("Historical Customer Support Cases:")
print(df)

Historical Customer Support Cases:
        Issue_Type       Product Priority Customer_Type  \
0    Login Problem    Mobile App     High       Regular   
1  Payment Failure  Subscription     High       Premium   
2    Login Problem    Web Portal   Medium       Regular   
3   Product Damage        Laptop     High       Premium   
4  Payment Failure    Mobile App   Medium       Regular   
5   Delivery Delay        Laptop      Low       Regular   
6   Product Damage    Mobile App   Medium       Premium   
7    Login Problem    Web Portal     High       Premium   

                                   Solution  
0         Reset password and verify account  
1  Retry payment and verify payment details  
2    Reset password and clear browser cache  
3                   Replace damaged product  
4     Verify card details and retry payment  
5        Track shipment and inform customer  
6              Initiate product replacement  
7         Reset password and verify account  


In [3]:
# ---------------------------------------------------------
# Step 2: Encode Categorical Attributes
# ---------------------------------------------------------

features = ["Issue_Type", "Product", "Priority", "Customer_Type"]

encoders = {}

for column in features:
    encoder = LabelEncoder()
    df[column + "_Encoded"] = encoder.fit_transform(df[column])
    encoders[column] = encoder

encoded_features = [column + "_Encoded" for column in features]
X = df[encoded_features].values

print("Encoded Features:")
print(df[encoded_features])

Encoded Features:
   Issue_Type_Encoded  Product_Encoded  Priority_Encoded  \
0                   1                1                 0   
1                   2                2                 0   
2                   1                3                 2   
3                   3                0                 0   
4                   2                1                 2   
5                   0                0                 1   
6                   3                1                 2   
7                   1                3                 0   

   Customer_Type_Encoded  
0                      1  
1                      0  
2                      1  
3                      0  
4                      1  
5                      1  
6                      0  
7                      0  


In [4]:
# ---------------------------------------------------------
# Step 3: Define New Customer Case
# ---------------------------------------------------------

new_case = {
    "Issue_Type": "Login Problem",
    "Product": "Web Portal",
    "Priority": "High",
    "Customer_Type": "Premium"
}

print("\nNew Customer Case:")
print(new_case)


New Customer Case:
{'Issue_Type': 'Login Problem', 'Product': 'Web Portal', 'Priority': 'High', 'Customer_Type': 'Premium'}


In [5]:
# ---------------------------------------------------------
# Step 4: Encode the New Case
# ---------------------------------------------------------

new_case_encoded = []

for column in features:
    value = encoders[column].transform([new_case[column]])[0]
    new_case_encoded.append(value)

new_case_vector = np.array(new_case_encoded).reshape(1, -1)

print("Encoded New Case:")
print(new_case_vector)

Encoded New Case:
[[1 3 0 0]]


In [6]:
# ---------------------------------------------------------
# Step 5: Calculate Euclidean Distance
# ---------------------------------------------------------

distances = euclidean_distances(X, new_case_vector).flatten()
df["Distance"] = distances

print("Euclidean Distances:")
print(df[features + ["Distance"]])

Euclidean Distances:
        Issue_Type       Product Priority Customer_Type  Distance
0    Login Problem    Mobile App     High       Regular  2.236068
1  Payment Failure  Subscription     High       Premium  1.414214
2    Login Problem    Web Portal   Medium       Regular  2.236068
3   Product Damage        Laptop     High       Premium  3.605551
4  Payment Failure    Mobile App   Medium       Regular  3.162278
5   Delivery Delay        Laptop      Low       Regular  3.464102
6   Product Damage    Mobile App   Medium       Premium  3.464102
7    Login Problem    Web Portal     High       Premium  0.000000


In [7]:
# ---------------------------------------------------------
# Step 6: Convert Distance into Similarity Score
# ---------------------------------------------------------

df["Similarity"] = 1 / (1 + df["Distance"])

print("Similarity Scores:")
print(df[features + ["Distance", "Similarity"]])

Similarity Scores:
        Issue_Type       Product Priority Customer_Type  Distance  Similarity
0    Login Problem    Mobile App     High       Regular  2.236068    0.309017
1  Payment Failure  Subscription     High       Premium  1.414214    0.414214
2    Login Problem    Web Portal   Medium       Regular  2.236068    0.309017
3   Product Damage        Laptop     High       Premium  3.605551    0.217129
4  Payment Failure    Mobile App   Medium       Regular  3.162278    0.240253
5   Delivery Delay        Laptop      Low       Regular  3.464102    0.224009
6   Product Damage    Mobile App   Medium       Premium  3.464102    0.224009
7    Login Problem    Web Portal     High       Premium  0.000000    1.000000


In [8]:
# ---------------------------------------------------------
# Step 7: Retrieve Most Similar Case
# ---------------------------------------------------------

best_index = df["Similarity"].idxmax()
best_case = df.loc[best_index]

print("\nMost Similar Historical Case:")
print(best_case[features])


Most Similar Historical Case:
Issue_Type       Login Problem
Product             Web Portal
Priority                  High
Customer_Type          Premium
Name: 7, dtype: object


In [9]:
# ---------------------------------------------------------
# Step 8: Display Recommendation
# ---------------------------------------------------------

print("\nSimilarity Score:", round(best_case["Similarity"], 4))

print("\nRecommended Solution:")
print(best_case["Solution"])


Similarity Score: 1.0

Recommended Solution:
Reset password and verify account


In [10]:
# ---------------------------------------------------------
# Step 9: CBR Cycle Interpretation
# ---------------------------------------------------------

print("\nCBR Cycle:")
print("1. Retrieve - Most similar historical case retrieved.")
print("2. Reuse - Previous solution recommended for the new case.")
print("3. Revise - Support staff can modify the solution if required.")
print("4. Retain - Resolved new case can be added to the case base.")


CBR Cycle:
1. Retrieve - Most similar historical case retrieved.
2. Reuse - Previous solution recommended for the new case.
3. Revise - Support staff can modify the solution if required.
4. Retain - Resolved new case can be added to the case base.
